<h1 style="text-align: center;">Week 1 - Gaussian Elimination </h1>

# Gaussian Elimination

## Background

Given a system of linear equations $A\mathbf{x} = \mathbf{b}$, where:
- $A$ is an $n \times n$ coefficient matrix
- $\mathbf{x}$ is the unknown vector
- $\mathbf{b}$ is the right-hand side vector

**Gaussian Elimination** solves for $\mathbf{x}$ by reducing the system into an equivalent upper triangular form, which is then easy to solve.
 
---

## Step 1: Augmented Matrix

Combine $A$ and $\mathbf{b}$ into a single augmented matrix:

$$[A \mid \mathbf{b}] = \begin{bmatrix} a_{11} & a_{12} & \cdots & a_{1n} & b_1 \\ a_{21} & a_{22} & \cdots & a_{2n} & b_2 \\ \vdots & \vdots & \ddots & \vdots & \vdots \\ a_{n1} & a_{n2} & \cdots & a_{nn} & b_n \end{bmatrix}$$

---

## Step 2: Forward Elimination

The goal is to reduce $[A \mid \mathbf{b}]$ to an **upper triangular matrix** $[U \mid \mathbf{b'}]$.

For each pivot row $k = 0, 1, \ldots, n-1$:

**a) Identify the Pivot Element**

The pivot is the diagonal element $a_{kk}$. Every subsequent elimination in column $k$ is relative to this element.

> ⚠️ **Note:** If the pivot element is zero (or near-zero), the matrix is **singular** and the system may have no unique solution.

**b) Eliminate Entries Below the Pivot**

For each row $i$ below the pivot row (i.e., $i > k$), compute the multiplier:

$$m_{ik} = \frac{a_{ik}}{a_{kk}}$$

Then update row $i$ by subtracting $m_{ik}$ times the pivot row:

$$\text{Row}_i \leftarrow \text{Row}_i - m_{ik} \times \text{Row}_k$$

This zeros out the entry at position $(i, k)$.

**c) Repeat**

Repeat steps (a) and (b) for each successive pivot until the matrix is **upper triangular**:

$$[U \mid \mathbf{b'}] = \begin{bmatrix} u_{11} & u_{12} & \cdots & u_{1n} & b'_1 \\ 0 & u_{22} & \cdots & u_{2n} & b'_2 \\ \vdots & \vdots & \ddots & \vdots & \vdots \\ 0 & 0 & \cdots & u_{nn} & b'_n \end{bmatrix}$$

---

## Step 3: Back Substitution

Once the system is upper triangular, solve for $\mathbf{x}$ from **bottom to top**:

$$x_n = \frac{b'_n}{u_{nn}}$$

$$x_i = \frac{b'_i - \displaystyle\sum_{j=i+1}^{n} u_{ij} \, x_j}{u_{ii}}, \quad i = n-1, \ldots, 0$$

---
---

In [248]:
import numpy as np

In [249]:
A = np.array([[2,4,-2],
             [4,9,-3],
             [-2,-3,-7]]) 

In [250]:
A 

array([[ 2,  4, -2],
       [ 4,  9, -3],
       [-2, -3, -7]])

In [251]:
# getting number of Rows
np.shape(A)[0]

3

In [252]:
b = np.array([[2],[8],[10]])

In [253]:
def forward_elimination(A, b):
    # creating an Augmented matrix 
    Au = np.hstack((A.astype(float),b.astype(float)))
    # using partial pivoting for elimination 
    # for each pivot we will swap the row having highest entry in the column and then eliminate entries below it
    
    rows = np.shape(A)[0] 
    
    
    for row in range(rows):
        max_pivot_row = np.argmax(np.abs(Au[row:,row]))+ row
        #switching the current row with the row having highest pivot element 
        if(max_pivot_row != row):
            Au[[row, max_pivot_row]] = Au[[max_pivot_row, row]]
        
        for i in range(row+1, rows):
            # pivot element 
            pivot = Au[row,row]
            # performing elimination 
            l = -(Au[i,row]/pivot)
            Au[i] = l*Au[row] + Au[i]
    
    return Au 

In [254]:
Au = forward_elimination(A,b)

In [255]:
def is_singular(Au, tol=1e-12):
    rows = np.shape(Au)[0]
    
    A_part = Au[:, :-1]
    max_abs_entry = np.max(np.abs(A_part))
    
    
    
    for i in range(rows):
        if abs(Au[i, i]) < tol * max_abs_entry:
            return 1 
    
    return 0 

In [256]:
is_singular(Au)

0

In [257]:
Au

array([[ 4.        ,  9.        , -3.        ,  8.        ],
       [ 0.        ,  1.5       , -8.5       , 14.        ],
       [ 0.        ,  0.        , -3.33333333,  2.66666667]])

In [258]:
def back_substitution(Au):
    n = np.shape(Au)[0]
    x = np.zeros((n))

    for i in range(n-1, -1, -1):
        x[i] = (Au[i,n] - np.dot(x,Au[i,:n]))/Au[i,i]
        
    return x

In [259]:
back_substitution(Au)

array([-9.4,  4.8, -0.8])

In [260]:
def gaussian_elimination(A,b):
    """
    returns the solution to the equation Ax = b, in case A is singular returns -1
    """
    #step 1 is to perform forward elimination 
    
    Au = forward_elimination(A, b)
    if(is_singular(Au)) == 1:
        return -1
    
    x = back_substitution(Au) 
    
    return x

In [261]:
gaussian_elimination(A,b)

array([-9.4,  4.8, -0.8])

In [262]:
#solving for another example 
A1 = np.array([[2,3,4],
             [1,1,1],
             [3,2,1]])

In [263]:
b1 = np.array([[20],[6],[13]])

In [264]:
gaussian_elimination(A1,b1)

-1

# Gauss-Jordan Method

## Background

While **Gaussian Elimination** reduces $A$ to an upper triangular matrix $U$ and relies on back substitution,
the **Gauss-Jordan Method** goes further — it eliminates entries **both below and above** each pivot,
reducing $A$ all the way to the **Identity matrix** $I$.

---

## Primary Use: Computing $A^{-1}$

The classic application is matrix inversion. By augmenting $A$ with the identity matrix and applying
Gauss-Jordan elimination:

$$[A \mid I] \xrightarrow{\text{Gauss-Jordan}} [I \mid A^{-1}]$$

---

## Solving $A\mathbf{x} = \mathbf{b}$ with Gauss-Jordan

Instead of augmenting with $I$, augment with $\mathbf{b}$ and apply the same process:

$$[A \mid \mathbf{b}] \xrightarrow{\text{Gauss-Jordan}} [I \mid \mathbf{x}]$$

The solution $\mathbf{x}$ appears directly in the last column — **no back substitution needed**.

---

## Steps

### Step 1: Form the Augmented Matrix

$$[A \mid \mathbf{b}] = \begin{bmatrix} a_{11} & a_{12} & \cdots & a_{1n} & b_1 \\ a_{21} & a_{22} & \cdots & a_{2n} & b_2 \\ \vdots & \vdots & \ddots & \vdots & \vdots \\ a_{n1} & a_{n2} & \cdots & a_{nn} & b_n \end{bmatrix}$$

### Step 2: Forward Elimination (same as Gaussian Elimination)

For each pivot row $k$, eliminate all entries **below** the pivot:

$$m_{ik} = \frac{a_{ik}}{a_{kk}}, \quad \text{Row}_i \leftarrow \text{Row}_i - m_{ik} \times \text{Row}_k \quad \forall \; i > k$$

This produces an upper triangular matrix $U$:

$$\begin{bmatrix} u_{11} & u_{12} & u_{13} & b'_1 \\ 0 & u_{22} & u_{23} & b'_2 \\ 0 & 0 & u_{33} & b'_3 \end{bmatrix}$$

### Step 3: Backward Elimination *(replaces Back Substitution)*

For each pivot row $k$ (now going **bottom to top**), eliminate all entries **above** the pivot:

$$m_{ik} = \frac{a_{ik}}{a_{kk}}, \quad \text{Row}_i \leftarrow \text{Row}_i - m_{ik} \times \text{Row}_k \quad \forall \; i < k$$

This zeroes out all off-diagonal entries, giving a diagonal matrix.

### Step 4: Scale Each Row

Divide each row $k$ by its pivot $a_{kk}$ to normalise the diagonal to 1:

$$\text{Row}_k \leftarrow \frac{\text{Row}_k}{a_{kk}}$$

Result:

$$\begin{bmatrix} 1 & 0 & 0 & x_1 \\ 0 & 1 & 0 & x_2 \\ 0 & 0 & 1 & x_3 \end{bmatrix}$$

The solution $\mathbf{x}$ is now directly readable from the last column.

---
---

In [265]:
def backward_elimination(Au):
    rows = np.shape(Au)[0]
    for i in range(rows-1,-1,-1):
        #dividing by the pivot to get the value
        Au[i] = Au[i]/Au[i,i]
        
        for j in range(i):
            multiplier = -(Au[j,i]/Au[i,i]) 
            Au[j] = multiplier * Au[i] + Au[j]
    
    return Au 

In [266]:
def gauss_jordan(A,b):
    """
    returns the solution to the equation Ax = b, in case A is singular returns -1
    """
    #step 1 is to perform forward elimination 
    
    Au = forward_elimination(A, b)
    if(is_singular(Au)) == 1:
        return -1
    
    Au = backward_elimination(Au) 
    
    return Au[:,-1]
    

In [267]:
gauss_jordan(A,b)

array([-9.4,  4.8, -0.8])

In [268]:
gauss_jordan(A1,b1)

-1

---
---

# Performance Comparison: Gaussian Elimination vs Gauss-Jordan

We use a **diagonally dominant** $1000 \times 1000$ matrix to guarantee:
- Non-singularity (unique solution exists)
- No zero pivots (no pivoting strategy needed)

$$a_{ii} = \sum_{j \neq i} |a_{ij}| + \delta, \quad \delta = 1.0$$


## Expected Behaviour

Gaussian Elimination performs:

$$\sim \frac{n^3}{3} \text{ flops for elimination} + \mathcal{O}(n^2) \text{ for back substitution}$$

Gauss-Jordan performs:

$$\sim \frac{n^3}{2} \text{ flops for elimination (no back substitution)}$$

Gauss-Jordan does approximately **50% more work** during elimination.
We expect:

$$t_{\text{Gauss-Jordan}} > t_{\text{Gaussian Elimination}}$$

---

In [269]:
np.random.seed(42)          # for reproducibility
A2 = np.random.randint(-9,10,(100, 100))

#making A2 diagonally dominant to avoid the non-singular case 
for i in range(np.shape(A2)[0]):
    A2[i,i] = np.sum(A2[i])

In [270]:
b2 = np.random.randint(-9,10,(100,1))

In [271]:
gaussian_elimination(A2,b2)

array([ 2.15726842,  0.30834893, -4.25750717,  0.75762568,  1.28120892,
       -3.99754872, -2.5417592 , -0.67623646, -0.65487313, -2.75691128,
        1.51180768,  2.09755472,  3.24448384, -1.63615828, -0.54513712,
       -0.1529921 ,  1.36008417,  1.53792975,  1.0625979 , -0.19597616,
        0.81205381,  0.63053663, -4.46376514, -2.4068343 , -0.2184591 ,
        1.53599762,  0.08905287,  3.25365046,  7.19951291, -0.33338766,
       -0.378977  ,  1.33701798,  0.97032923,  0.59652694,  2.96477811,
        0.80524964, -5.05525783,  0.66265144, -0.88991312,  3.97609321,
       -0.61906459, -0.25606743, -2.58082905, -3.5528156 ,  0.92900803,
       -2.31080958,  0.48262685,  1.56744436,  4.05750803, -0.66249929,
       -1.24554085, -0.23691086,  1.45875854, -2.14432677, -2.33334183,
       -3.67202074,  3.36733171,  1.09934445,  0.92037739,  3.22161805,
        4.47798093,  0.70299986,  1.0447481 , -1.64063707,  2.53133735,
       -1.22018781, -0.29290637, -0.13949545, -4.12471381,  1.67

In [272]:
gauss_jordan(A2,b2)

array([ 2.15726842,  0.30834893, -4.25750717,  0.75762568,  1.28120892,
       -3.99754872, -2.5417592 , -0.67623646, -0.65487313, -2.75691128,
        1.51180768,  2.09755472,  3.24448384, -1.63615828, -0.54513712,
       -0.1529921 ,  1.36008417,  1.53792975,  1.0625979 , -0.19597616,
        0.81205381,  0.63053663, -4.46376514, -2.4068343 , -0.2184591 ,
        1.53599762,  0.08905287,  3.25365046,  7.19951291, -0.33338766,
       -0.378977  ,  1.33701798,  0.97032923,  0.59652694,  2.96477811,
        0.80524964, -5.05525783,  0.66265144, -0.88991312,  3.97609321,
       -0.61906459, -0.25606743, -2.58082905, -3.5528156 ,  0.92900803,
       -2.31080958,  0.48262685,  1.56744436,  4.05750803, -0.66249929,
       -1.24554085, -0.23691086,  1.45875854, -2.14432677, -2.33334183,
       -3.67202074,  3.36733171,  1.09934445,  0.92037739,  3.22161805,
        4.47798093,  0.70299986,  1.0447481 , -1.64063707,  2.53133735,
       -1.22018781, -0.29290637, -0.13949545, -4.12471381,  1.67

In [273]:
%timeit gaussian_elimination(A2,b2)

39.6 ms ± 2.68 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [274]:
%timeit gauss_jordan(A2,b2)

73 ms ± 2.48 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [275]:
#increasing the matrix size to 1000x1000 
#we will not be seeing the solution, we will just see the time taken by the two methods 

In [276]:
np.random.seed(42)          # for reproducibility
A3 = np.random.randint(-9,10,(1000, 1000))

#making A2 diagonally dominant to avoid the singular case 
for i in range(np.shape(A3)[0]):
    A3[i,i] = np.sum(A3[i])

In [277]:
b3 = np.random.randint(-9,10,(1000,1))

In [278]:
%timeit gaussian_elimination(A3,b3)

5.8 s ± 136 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [279]:
%timeit gauss_jordan(A3,b3)

12.1 s ± 383 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


<span style="color: blue;">
<b>It can be seen that Gauss Elimination takes ~50% lower time than Gauss Jordan Method to solve the equationn Ax = b</b>

</span>

---

---

# Built-in linear algebra funtion to solve Ax = b

In [287]:
x = np.linalg.solve(A, b)

In [288]:
x

array([[-9.4],
       [ 4.8],
       [-0.8]])

In [289]:
%timeit np.linalg.solve(A3,b3)

100 ms ± 24.4 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


---
---